# Exp8.0.2 — Joint L1/L2 Supervision

Analysis-only notebook. Training/evaluation happens in the Slurm array; this notebook reads finalized artifacts only.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
while repo.name != 'writingRing' and repo.parent != repo:
    repo = repo.parent
base = repo / 'notebooks' / 'artifacts' / 'experiment_8_0_2_joint_l1_l2_supervision' / 'joint_l1_l2_supervision_v1'
manifest = json.loads((base / 'manifest.json').read_text())
method_runs = pd.read_csv(base / 'method_runs.csv')
method_summary = pd.read_csv(base / 'method_summary.csv')
probe_runs = pd.read_csv(base / 'probe_runs.csv')
probe_summary = pd.read_csv(base / 'probe_summary.csv')
fusion_runs = pd.read_csv(base / 'fusion_gain_runs.csv')
fusion_summary = pd.read_csv(base / 'fusion_gain_summary.csv')
overlap_summary = pd.read_csv(base / 'correctness_overlap_summary.csv')
coef_summary = pd.read_csv(base / 'coef_block_summary.csv')
head_summary = pd.read_csv(base / 'trained_head_summary.csv')
history_runs = pd.read_csv(base / 'history_runs.csv')
manifest

## Native readout and same-W LIF transfer

In [ ]:
cols = ['method', 'native_test_ba_mean', 'native_test_ba_std', 'lif_test_ba_mean', 'lif_test_ba_std', 'lif_penalty_mean', 'parameter_count_mean']
display(method_summary[cols])
plot_df = method_runs.groupby('method')[['native_test_ba', 'lif_test_ba']].mean()
ax = plot_df.plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_ylabel('Balanced accuracy')
ax.set_title('Native Linear vs same-W output LIF')
plt.tight_layout()

## Frozen L1/L2 representation probes

In [ ]:
key_features = ['l1_whole', 'l2_whole', 'l1_l2_whole', 'l1_fixed250', 'l2_fixed250', 'l1_l2_fixed250']
view = probe_summary[probe_summary.feature.isin(key_features)][['method', 'feature', 'test_ba_mean', 'test_ba_std', 'train_test_gap_mean']]
display(view.pivot(index='method', columns='feature', values='test_ba_mean'))
pivot = probe_runs[probe_runs.feature.isin(key_features)].groupby(['method', 'feature']).test_ba.mean().unstack()
ax = pivot.plot(kind='bar', figsize=(12, 5), rot=0)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Frozen representation probes')
plt.tight_layout()

## Does training create L1/L2 complementarity?

In [ ]:
display(fusion_summary)
gain_cols = ['whole_fusion_gain_over_best_single', 'fixed250_fusion_gain_over_best_single']
gain = fusion_runs.groupby('method')[gain_cols].mean()
ax = gain.plot(kind='bar', figsize=(9, 4), rot=0)
ax.axhline(0, linewidth=1)
ax.set_ylabel('Fusion gain in BA')
ax.set_title('Post-hoc L1+L2 gain over best single layer')
plt.tight_layout()

## Correctness overlap and oracle union

In [ ]:
test_overlap = overlap_summary[(overlap_summary.split == 'test')][['method', 'aggregation', 'l1_only_correct_mean', 'l2_only_correct_mean', 'prediction_disagreement_mean', 'oracle_union_accuracy_mean']]
display(test_overlap)

## Trained-head and probe block usage

In [ ]:
display(head_summary)
joint_blocks = coef_summary[coef_summary.feature.isin(['l1_l2_whole', 'l1_l2_fixed250'])][['method', 'feature', 'layer', 'coef_rms_mean', 'coef_norm_fraction_mean']]
display(joint_blocks)

## Mean training curves across seeds

In [ ]:
curve = history_runs.groupby(['method', 'epoch'], as_index=False)[['train_ba', 'val_ba']].mean()
fig, ax = plt.subplots(figsize=(10, 5))
for method, frame in curve.groupby('method'):
    ax.plot(frame.epoch, frame.val_ba, label=f'{method} val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Validation BA vs epoch')
ax.legend()
plt.tight_layout()

curve_loss = history_runs.groupby(['method', 'epoch'], as_index=False)[['train_loss', 'val_loss']].mean()
fig, ax = plt.subplots(figsize=(10, 5))
for method, frame in curve_loss.groupby('method'):
    ax.plot(frame.epoch, frame.val_loss, label=f'{method} val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Objective loss')
ax.set_title('Validation objective loss vs epoch')
ax.legend()
plt.tight_layout()

## Interpretation checklist

1. Does `l1_l2_joint` improve native BA over `l2_only` across paired seeds?
2. Does frozen `L1+L2` probe gain become consistently positive after joint supervision?
3. Does `l2_main_l1_aux` improve L2-only representation while keeping inference L2-only?
4. Do L1-only-correct/L2-only-correct fractions increase or become more exploitable?
5. Does the trained joint head actually use both heads, or collapse onto one?